# Running LLMs on a Laptop: GGUF and llama.cpp

The full code of [Running LLMs on a Laptop: GGUF and llama.cpp](https://residual-stream.lvtan71.workers.dev/posts/running-llms-on-a-laptop/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/running-llms-on-a-laptop) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/running-llms-on-a-laptop.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 1 min end to end.
The post's speed numbers are from llama.cpp on Apple's Metal GPU; on Colab, llama-cpp-python runs on the CPU (it takes several minutes to install), so expect the quality numbers to match and the speeds not to.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q llama-cpp-python gguf huggingface_hub numpy

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "running-llms-on-a-laptop"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

In [ ]:
!tar xzf blog-ed51ca8.tar.gz

## `gguf_contents.py`

What is inside a GGUF file: Qwen2.5-1.5B-Instruct Q4_K_M's metadata, its tensors grouped by quantization type, which tensors are kept at Q6_K, and the average bits per weight of the Q3_K_M, Q4_K_M and Q8_0 files.

In [ ]:
!{sys.executable} gguf_contents.py

The post printed:

```text
qwen2.5-1.5b-instruct-q4_k_m.gguf: 26 metadata keys, 339 tensors
  general.architecture             qwen2
  qwen2.block_count                28
  qwen2.context_length             32768
  qwen2.embedding_length           1536
  qwen2.attention.head_count       12
  qwen2.attention.head_count_kv    2
  general.file_type                15
  tokenizer.ggml.tokens            151,936 entries (the vocabulary, in the file)
tensors by type:
  Q4_K    169 tensors   1345.4M weights    721.7 MiB   4.50 bits per weight
  Q6_K     29 tensors    431.6M weights    337.6 MiB   6.56 bits per weight
  F32     141 tensors      0.1M weights      0.6 MiB  32.00 bits per weight
examples: output.weight Q6_K, token_embd.weight Q4_K, blk.0.attn_norm.weight F32, blk.0.ffn_down.weight Q6_K, blk.0.ffn_gate.weight Q4_K, blk.0.ffn_up.weight Q4_K
the 29 Q6_K tensors: output.weight, 14 attn_v and 14 ffn_down (layers [0, 1, 5, 6, 7, 8, 9, 10, 13, 16, 19, 21, 24, 27])
qwen2.5-1.5b-instruct-q3_k_m.gguf     0.92 GB,  4.13 bits per weight on average
qwen2.5-1.5b-instruct-q4_k_m.gguf     1.12 GB,  5.00 bits per weight on average
qwen2.5-1.5b-instruct-q8_0.gguf       1.89 GB,  8.50 bits per weight on average
```

## `speed.py`

Decode speed of the three files on the GPU (all layers offloaded) and on 8 CPU threads, median of 3 runs of 128 tokens, against the bandwidth bound of 93 GB/s.

In [ ]:
!{sys.executable} speed.py

The post printed:

```text
file                                 GB read/token     bound  GPU (Metal)  of bound  CPU, 8 threads
qwen2.5-1.5b-instruct-q8_0.gguf             1.64       57/s        57/s     101%           62/s
qwen2.5-1.5b-instruct-q4_k_m.gguf           0.98       95/s        86/s      90%           90/s
qwen2.5-1.5b-instruct-q3_k_m.gguf           0.82      114/s        73/s      64%           85/s
```

## `quality.py`

Bits per token of each file on two of the blog's posts, from llama.cpp's own logits.

In [ ]:
!{sys.executable} quality.py

The post printed:

```text
token_embd shape in the file: [np.uint64(1536), np.uint64(151936)]
qwen2.5-1.5b-instruct-q8_0.gguf      3.061 bits per token
qwen2.5-1.5b-instruct-q4_k_m.gguf    3.114 bits per token
qwen2.5-1.5b-instruct-q3_k_m.gguf    3.185 bits per token
```